# Prompt Engineering

In [2]:
import os

import anthropic
import ollama
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv
from openai import OpenAI, http_client


In [3]:
# constants
LLM_PROVIDER_CHATGPT = "ChatGPT"
LLM_PROVIDER_OLLAMA = "Ollama"
LLM_PROVIDER_CLAUDE = "Claude"

selected_llm_provider = LLM_PROVIDER_OLLAMA

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'mistral:7b'
MODEL_CLAUDE = 'claude-3-5-haiku-20241022'

# Initialize and set up environment
load_dotenv(override=True)

#os.environ["HUGGINGFACEHUB_API_TOKEN"]

# Attempt to create the OpenAI client object
try:
    openai_api_key = os.getenv('OPENAI_API_KEY')
    openai_client = OpenAI(api_key=openai_api_key)
    print("OpenAI client object created successfully.")
except Exception as e:
    print(f"Error creating OpenAI client object: {e}")

# Attempt to create the Anthropic client object
try:
    anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
    anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
    print("Anthropic client object created successfully.")
except Exception as e:
    print(f"Error creating Anthropic client object: {e}")


OpenAI client object created successfully.
Anthropic client object created successfully.


In [4]:
def build_messages(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    return messages

def build_messages_for_claude(user_prompt):
    messages = [
        {"role": "user", "content": user_prompt}
    ]
    return messages


In [6]:
def ask_chatgpt(messages):
    stream = openai_client.chat.completions.create(model=MODEL_GPT, messages=messages,stream=True)

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(f"Answer: {response}"), display_id=display_handle.display_id)

def ask_ollama(messages):
    stream = ollama.chat(
        model = MODEL_LLAMA,
        messages = messages,
        stream = True
    )

    display_handle = display(Markdown(""), display_id=True)
    response = ""

    for chunk in stream:
        #print(chunk['message']['content'], end='', flush=True)
        content = chunk['message']['content']
        response += content
        update_display(Markdown(f"Answer: {response}"), display_id=display_handle.display_id)

def ask_claude(system_prompt, messages):
    stream = anthropic_client.messages.create(
        model=MODEL_CLAUDE,
        max_tokens=1000,
        system=system_prompt,
        messages=messages,
        stream=True,
        temperature=1
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        # Check if the chunk is a ContentBlockDeltaEvent
        if chunk.type == "content_block_delta":
            #if hasattr(chunk, 'delta') and hasattr(chunk.delta, 'text'):
            response += chunk.delta.text
            update_display(Markdown(f"Answer: {response}"),  display_id=display_handle.display_id)


In [7]:
def ask_llm():
    display(Markdown(f"# *Selected LLM:* ***{selected_llm_provider}***"))

    if selected_llm_provider == LLM_PROVIDER_CHATGPT:
        messages = build_messages(system_prompt, user_prompt)
        ask_chatgpt(messages)
    elif selected_llm_provider == LLM_PROVIDER_OLLAMA:
        messages = build_messages(system_prompt, user_prompt)
        ask_ollama(messages)
    elif selected_llm_provider == LLM_PROVIDER_CLAUDE:
        messages = build_messages_for_claude(user_prompt)
        ask_claude(system_prompt, messages)


## Provide clear instructions

In [10]:
system_prompt = """
You are an AI assistant that helps human by generating tutorials given a text.
You will be provided with a text. If the text contains any kind of istructions on how to proceed with something, generate a tutorial in a bullet list.
Otherwise, inform the user that the text does not contain any instructions.

Text:
"""

user_prompt = """
To prepare the known sauce from Genova, Italy, you can start by toasting the pine nuts to then coarsely
chop them in a kitchen mortar together with basil and garlic. Then, add half of the oil in the kitchen mortar and season with salt and pepper.
Finally, transfer the pesto to a bowl and stir in the grated Parmesan cheese.

"""

In [11]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  **Tutorial: How to Prepare Genova Sauce**

1. Toast the pine nuts until they are golden brown.
2. Coarsely chop the toasted pine nuts, basil, and garlic in a kitchen mortar.
3. Add half of the oil into the kitchen mortar.
4. Season the mixture with salt and pepper.
5. Transfer the pesto from the kitchen mortar to a bowl.
6. Stir in the grated Parmesan cheese into the pesto in the bowl.

Enjoy your homemade Genova Sauce!

## Split complext tasks into subtasks

In [8]:
system_prompt = """
You are an AI assistant that summarize articles. 
To complete this task, do the following subtasks:

Read the provided article context comprehensively and identified the main topic and key points
Generated a paragraph summary of the current article context that captures the essential information and conveys the main idea
Print each step of the proces.
Article:
"""

user_prompt = """
Recurrent neural networks, long short-term memory and gated recurrent neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures.
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht-1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through factorization tricks and conditional
computation, while also improving model performance in case of the latter. The fundamental
constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences. In all but a few cases, however, such attention mechanisms
are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for significantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
"""


In [9]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  **Step 1: Reading the provided article context comprehensively**
The article discusses the use of Recurrent Neural Networks (RNN), Long Short-Term Memory (LSTM), and Gated Recurrent Neural Networks (GRNN) in sequence modeling and transduction problems like language modeling and machine translation. It highlights that these models compute based on the symbol positions of input and output sequences, which leads to sequential computation that cannot be parallelized within training examples. This becomes a problem at longer sequence lengths due to memory constraints limiting batching across examples.

The article also mentions that attention mechanisms have become essential for sequence modeling tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences. However, these attention mechanisms are typically used in conjunction with a recurrent network.

**Step 2: Identifying the main topic and key points**
The main topic of this article is an introduction to the Transformer, a new model architecture that eschews recurrence and instead relies entirely on an attention mechanism to draw global dependencies between input and output. The key points are that the Transformer allows for significantly more parallelization, can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs, and it does not use recurrence like traditional RNN, LSTM, or GRNN models.

**Step 3: Generated a paragraph summary**
The article introduces the Transformer, a new model architecture that relies solely on an attention mechanism to draw global dependencies between input and output. This architecture eliminates the need for recurrence found in traditional RNN, LSTM, or GRNN models, allowing for significantly more parallelization and reaching a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.

## Ask for justification

In [10]:
system_prompt = """
You are an AI assistant specialized in generating essays shorter than 500 words.
Given a statement, develop the essay the best you can.
Once the essay is generated, provide clear justifications and explanations of the reasons behind the sentences you generated.

Statement:

"""

user_prompt = """
An introduction to mammals like chickens.
"""

In [11]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  Title: An Introduction to Mammals: Focusing on Domestic Chickens

In the vast and diverse kingdom of animals, a significant portion is dedicated to mammals. Among these, one of the most common and utilized species worldwide is the chicken (Gallus gallus domesticus). This essay aims to provide an introduction to mammals, focusing specifically on chickens.

Chickens belong to the class Mammalia, which distinguishes them from other vertebrates due to their unique characteristics. One of the defining features of mammals is their hair or fur covering, a trait that chickens lack. Instead, they possess a layer of feathers that provide insulation against cold temperatures and serve as camouflage in their environments.

Domestic chickens are omnivores, meaning they consume a diet that includes both plants (grains, fruits, vegetables) and animals (insects, worms). This adaptability has allowed them to thrive in various habitats, from farms to backyards, and even in the wild.

One of the most fascinating aspects of chickens is their reproduction. Females, or hens, lay eggs that are a rich source of protein. The hen incubates these eggs for approximately 21 days until they hatch into tiny chicks. Unlike many mammals, chickens do not carry their offspring in their bodies during pregnancy.

Chickens have been domesticated for thousands of years and play crucial roles in human societies. They are raised primarily for their eggs, meat, and feathers. In addition, they serve as indicators of environmental changes due to their sensitivity to certain conditions such as soil quality and air pollution.

In conclusion, chickens are fascinating examples of mammals adapted to human environments. Their omnivorous diet, unique physical characteristics, and reproductive process set them apart within the Mammalia class. Understanding chickens provides valuable insights into the broader world of mammals and their role in our lives.

Justification:
1. Introduction: Provides context by introducing the topic and stating the focus.
2. Defining features: Explains what sets mammals apart from other vertebrates, using a relatable example (chickens).
3. Diet: Describes the dietary habits of chickens to illustrate their adaptability.
4. Habitat: Mentions various habitats where chickens thrive, highlighting their versatility.
5. Reproduction: Explains chicken reproduction in a way that distinguishes them from most mammals.
6. Importance to humans: Discusses the roles of chickens in human societies and their environmental significance.
7. Conclusion: Summarizes the main points and emphasizes the importance of understanding chickens within the context of mammals.

In [12]:
system_prompt = """
You are an AI assistant specialized in solving riddles.
Given a riddle, solve it the best you can.
Provide a clear justification of your answer and the reasoning behind it.

Riddle:

"""

user_prompt = """
What has a face and two hands, but no arms or legs?
"""

In [13]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  The answer to this riddle is "A clock."

Justification: A clock has a face that represents the hours and minutes, and it has two hands (one short for minutes and one long for hours) to show the time. However, a clock does not have arms or legs like a human or any other animal. The riddle uses this ambiguity in our mental image of 'hands' to misdirect us initially, but once we realize that the 'hands' are parts of the clock, it becomes clear that a clock is the correct answer.

## Generate many output - self consistency

In [14]:
system_prompt = """
You are an AI assistant specialized in solving riddles.
Given a riddle, you have to generate three answers to the riddle.
For each answer, be specific about the reasoning you made.
Then, among the three answer, select the one which is most plausible given the riddle.

Riddle:

"""

user_prompt = """
There are four friends: Alice, Bob, Charlie, and David. They each have a different favorite color: red, blue, green, and yellow. They also each have a different favorite animal: cat, dog, fish, and bird. Using the following clues, can you figure out who likes what color and what animal?

Alice does not like red or blue.
Bob likes dogs, but not green.
Charlie likes fish, but not yellow.
David likes yellow, but not cats.
The person who likes red also likes birds.
The person who likes blue also likes cats.

You can write your answer in the form of four sentences, such as “Alice likes green and fish.” 

"""



In [15]:
system_prompt = """
You are an AI assistant specialized in solving riddles.
Given a riddle, you have to generate three answers to the riddle.
For each answer, be specific about the reasoning you made.
Then, among the three answer, select the one which is most plausible given the riddle.

Riddle:

"""

user_prompt = """
What has a face and two hands, but no arms or legs?

"""



In [16]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  Answer 1: The answer could be a clock. Reasoning: A clock has a face to display the time, and it has two hands (the hour hand and the minute hand) that move around the face, but clocks do not have arms or legs.

Answer 2: Another possible answer is a statue. Statues often have detailed faces and limbs represented in the form of arms and legs, but real-world statues do not actually have arms or legs as they are inanimate objects.

Answer 3: The third answer could be a glove puppet. Puppets are designed to resemble animals or characters, with a face and two hands for the user to manipulate. Unlike living beings, glove puppets do not have arms or legs of their own.

Most Plausible Answer: Based on the riddle provided, the most plausible answer is a clock. While a statue and a glove puppet could technically fit the description, clocks are more commonly associated with having faces and two hands, but no arms or legs. Additionally, the question refers to something that doesn't have arms or legs, which is not as straightforward for statues or glove puppets because they are often intentionally designed to resemble creatures with arms and/or legs.

## Order Matters

In [17]:
system_prompt = """
You are a sentiment analyzer. You classify conversations into three categories: positive, negative or neutral.
Return only the sentiment, in lower cap and without punctuation.

Conversation:

"""

user_prompt = """
Customer: Hi, I need some help with my order.
AI agent: Hello, welcome to our online store. I'm an AI agent and I'm here to assist you. 
Customer: I ordered a pair of shoes yesterday, but I haven't received a confirmation email yet. Can you check the status of my order?
AI agent: Sure, I can help you with that. Can you please provide me with your order number and email address?
Customer: Yes, my order number is 123456789 and my email is john.doe@example.com.
AI agent: Thank you. I have found your order in our system. It looks like your order is still being processed and it will be shipped soon. You should receive a confirmation email within 24 hours.
Customer: OK, thank you for the information. How long will it take for the shoes to arrive?
AI agent: You're welcome. According to our shipping policy, it will take about 3 to 5 business days for the shoes to arrive at your address. You can track your order online using the tracking number that will be sent to your email once your order is shipped.
Customer: Alright, sounds good. Thank you for your help.
AI agent: It's my pleasure. Is there anything else I can do for you today?
Customer: No, that's all. Have a nice day.
AI agent: Thank you for choosing our online store. Have a nice day too. Goodbye. 
"""

In [18]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  neutral

In [19]:
system_prompt = f"""
You are a sentiment analyzer. You classify conversations into three categories: positive, negative or neutral.
Return only the sentiment, in lower cap and without punctuation.

Conversation:
{user_prompt}

Remember to return only the sentiment, in lower cap and without punctuation!

"""

In [20]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  neutral

In [21]:
user_prompt = """
A neutron is a subatomic particle that has a neutral (not positive or negative) charge, and a mass slightly greater than that of a proton. 
It is present in all atomic nuclei except those of ordinary hydrogen. 
Neutrons, along with protons and electrons, are one of the three basic particles making up atoms. 
The term “neutron” comes from the fact that it is electrically neutral, meaning it carries no charge.
"""

system_prompt = f"""
Reframe the text for a 5 years old child. It should be shorter than 500 words. Make a parallelism with animals.

The text is the following:

{user_prompt}

"""



In [22]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  Meet our friend Neutron, a tiny superhero in the world of atoms! Unlike Electron (who has a negative charge) and Proton (who has a positive one), Neutron doesn't have any charge at all - he's just like a regular cat or dog without fur!

Neutrons can be found inside all atomic homes, except those of simple hydrogen houses. Just like how many cats live in apartments but not in small shacks, neutrons aren't around when it comes to ordinary hydrogen.

Now here's the fun part: Neutrons are one of the three main building blocks for making atoms! Protons and electrons are the other two, much like how bricks, windows, and doors make a house.

The name 'neutron' comes from its special feature - it doesn't carry any electricity! Just as we know that an elephant doesn't have wings, similarly, we know that neutrons don't have charges. So whenever you see Neutron in your atomic adventure book, remember he's the neutral one!

## Use delimiters

In [23]:
system_prompt = """
You are a Python expert that produces python code as per user's request.

===>START EXAMPLE

---User Query---
Give me a function to print a string of text.

---User Output---
Below you can find the described function:
```def my_print(text):
     return print(text)
```
<===END EXAMPLE
"""

user_prompt = "generate a python function to calculate the nth Fibonacci number"

In [24]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  Below you can find the described function for calculating the nth Fibonacci number:

```python
def fibonacci(n):
    if n <= 0:
        raise ValueError("Argument must be a positive integer")

    fib = [0, 1]

    # Calculate the rest of the Fibonacci sequence
    for i in range(2, n + 1):
        fib.append(fib[i - 1] + fib[i - 2])

    return fib[-1]
```

This function calculates the nth Fibonacci number using a loop and storing the results in a list to avoid recalculation of already known numbers. It also handles cases where `n` is not a positive integer by raising a ValueError.

## Few shot learning

In [25]:
system_prompt = """
You are an AI marketing assistant. You help users to create taglines for new product names.
Given a product name, produce a tagline similar to the following examples:

Peak Pursuit - Conquer Heights with Comfort
Summit Steps - Your Partner for Every Ascent
Crag Conquerors - Step Up, Stand Tal

Product name:

"""

user_prompt = 'Elevation Embrace'

In [26]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  Elevation Embrace - Reach New Heights, Together in Comfort

In [27]:

import pandas as pd

df = pd .read_csv('movie.csv', encoding='utf-8')
df['label'] = df['label'].replace({0: 'Negative', 1: 'Positive'})
df.head()

,text,label
0,I grew up (b. 1965) watching and loving the Th...,Negative
1,"When I put this movie in my DVD player, and sa...",Negative
2,Why do people who do not know what a particula...,Negative
3,Even though I have great interest in Biblical ...,Negative
4,Im a die hard Dads Army fan and nothing will e...,Positive


In [28]:
df = df.sample(n=10, random_state=42)  # Change the value of 'random_state' as needed for reproducibility
df.head()


,text,label
32823,The central theme in this movie seems to be co...,Negative
16298,"An excellent example of ""cowboy noir"", as it's...",Positive
28505,The ending made my heart jump up into my throa...,Negative
6689,Only the chosen ones will appreciate the quali...,Positive
26893,"This is a really funny film, especially the se...",Positive


In [29]:
system_prompt = """
You are a binary classifier for sentiment analysis.
Given a text, based on its sentiment you classify it into one of two categories: positive or negative.

You can use the following texts as examples:

Text: "I love this product! It's fantastic and works perfectly."
Positive

Text: "I'm really disappointed with the quality of the food."
Negative

Text: "This is the best day of my life!"
Positive

Text: "I can't stand the noise in this restaurant."
Negative

ONLY return the sentiment as output (without punctuation).

Text:

"""

user_prompt = "The concert was amazing! The band was incredible!"

In [30]:
ask_llm()

# *Selected LLM:* ***Ollama***

Answer:  Positive

In [31]:
def process_text(text):
    response = ollama.chat(
        model = MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ]
    )
    # Alternatively, access fields directly from the response object: print(response.message.content)
    return response['message']['content']

df['predicted'] = df['text'].apply(process_text)

print(df)


                                                    text     label  predicted
32823  The central theme in this movie seems to be co...  Negative   Negative
16298  An excellent example of "cowboy noir", as it's...  Positive   Positive
28505  The ending made my heart jump up into my throa...  Negative   Positive
6689   Only the chosen ones will appreciate the quali...  Positive   Positive
26893  This is a really funny film, especially the se...  Positive   Positive
36572  Sure, we all like bad movies at one time or an...  Negative   Negative
12335  Why?!! This was an insipid, uninspired and emb...  Negative   Negative
29591  This is one of those movies that has everythin...  Positive   Positive
18948  i saw this film over 20 years ago and still re...  Positive   Positive
31067  This true story of Carlson's Raiders is more o...  Negative   Negative


In [33]:
df.head()

,text,label,predicted
32823,The central theme in this movie seems to be co...,Negative,Negative
16298,"An excellent example of ""cowboy noir"", as it's...",Positive,Positive
28505,The ending made my heart jump up into my throa...,Negative,Positive
6689,Only the chosen ones will appreciate the quali...,Positive,Positive
26893,"This is a really funny film, especially the se...",Positive,Positive


## CoT

In [34]:
system_prompt = """
To solve a generic first-degree equation, follow these steps:

1. **Identify the Equation:** Start by identifying the equation you want to solve. It should be in the form of "ax + b = c," where 'a' is the coefficient of the variable, 'x' is the variable, 'b' is a constant, and 'c' is another constant.

2. **Isolate the Variable:** Your goal is to isolate the variable 'x' on one side of the equation. To do this, perform the following steps:
   
   a. **Add or Subtract Constants:** Add or subtract 'b' from both sides of the equation to move constants to one side.
   
   b. **Divide by the Coefficient:** Divide both sides by 'a' to isolate 'x'. If 'a' is zero, the equation may not have a unique solution.

3. **Simplify:** Simplify both sides of the equation as much as possible.

4. **Solve for 'x':** Once 'x' is isolated on one side, you have the solution. It will be in the form of 'x = value.'

5. **Check Your Solution:** Plug the found value of 'x' back into the original equation to ensure it satisfies the equation. If it does, you've found the correct solution.

6. **Express the Solution:** Write down the solution in a clear and concise form.

7. **Consider Special Cases:** Be aware of special cases where there may be no solution or infinitely many solutions, especially if 'a' equals zero.


Equation:

"""

equation = "3x + 5 = 11"

In [35]:
response = ollama.chat(
    model = MODEL_LLAMA,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": equation},
    ]
)

print(response.message.content)


 Let's solve the given equation step by step:

1. **Identify the Equation:** The equation is "3x + 5 = 11."

2. **Isolate the Variable:** We want to isolate 'x.'
   a. Subtract 5 from both sides: "3x - 5 = 11 - 5"
   b. Simplify: "3x = 6"
   c. Divide by 3: "x = 6/3"

3. **Simplify:** The solution is already simplified: "x = 2"

4. **Solve for 'x':** We found that 'x' equals 2.

5. **Check Your Solution:** Plug the found value of 'x' back into the original equation to ensure it satisfies the equation: "3*2 + 5 = 11." This is true, so we have the correct solution.

6. **Express the Solution:** Write down the solution in a clear and concise form: "The solution is x = 2"

7. **Consider Special Cases:** In this case, there are no special cases to consider as 'a' (coefficient of x) is not zero. The equation has one unique solution.


## ReAct

In [1]:
# original version with deprecated classes

import os
from dotenv import load_dotenv
from langchain_community.utilities.serpapi import SerpAPIWrapper

from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
from langchain_classic.tools import Tool
from langchain_openai import OpenAI

load_dotenv()

# 1. Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/react")

# 2. Choose the LLM to use
# Attempt to create the OpenAI client object
try:
    openai_api_key = os.getenv('OPENAI_API_KEY')
    openai_client = OpenAI(api_key=openai_api_key)
    print("OpenAI client object created successfully.")

    key = os.environ["SERPAPI_API_KEY"]
    search = SerpAPIWrapper()
    tools = [ Tool.from_function(
        func=search.run,
        name="Search",
        description="useful for when you need to answer questions about current events"
    )] # Add your tools here

    # 3. Construct the ReAct agent
    agent = create_react_agent(openai_client, tools, prompt)

    # 4. Create an agent executor by passing in the agent and tools
    agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

    # 5. Run
    agent_executor.invoke({"input": "What is 2 + 2?"})
except Exception as e:
    print(f"Error on running agent object: {e}")



OpenAI client object created successfully.


> Entering new AgentExecutor chain...
 I should use my knowledge of basic math to solve this question.
Action: Search
Action Input: "2 + 2"44 is the answer to the equation.
Final Answer: 4

> Finished chain.


In [2]:
print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [3]:
agent_executor.invoke({"input": "who are going to be the italian male athletes for climbing at the Paris 2024 Olympics?"})



> Entering new AgentExecutor chain...
 I should search for information about the Italian male athletes for climbing at the Paris 2024 Olympics.
Action: Search
Action Input: "Italian male athletes climbing Paris 2024 Olympics"['Italy competed at the 2024 Summer Olympics in Paris from 26 July to 11 August 2024. Italian athletes have appeared in every Summer Olympics edition of the ...', 'Sport Climbing · Ludovico Fossali, Deng Lijuan claim speed climbing titles at IFSC World Cup Briançon · Paris 2024 | Olympic Games · Olympic Games · Milano ...', 'Matteo ZURLONI ; Games Participations1 ; First Olympic GamesParis 2024 ; Year of Birth2002.', "The Paris 2024 line-up is now complete after Speed athletes were confirmed yesterday and the men's and women's Boulder & Lead athletes were confirmed today.", 'The competitions in the disciplines of speed and bouldering & lead will take place from August 5 to 10. The following 68 climbers are taking part.', 'Superlative performance also for Matteo Z

{'input': 'who are going to be the italian male athletes for climbing at the Paris 2024 Olympics?',
 'output': 'The Italian male athletes for climbing at the Paris 2024 Olympics are Ludovico Fossali, Matteo Zurloni, Stefano Ghisolfi, and Leonardo.'}

In [1]:
# modern implementation using LangGraph with ChatOpenAI

import os
from dotenv import load_dotenv

# Modern Imports
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool
from langchain.agents import create_agent

load_dotenv()

try:
    # 1. Choose the LLM to use
    # We use ChatOpenAI as it supports the modern tool-calling features required by create_agent
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    print("OpenAI Chat client object created successfully.")

    # 2. Define Tools
    if "SERPAPI_API_KEY" not in os.environ:
        raise ValueError("SERPAPI_API_KEY is missing from environment variables")

    search = SerpAPIWrapper()

    tools = [
        Tool(
            name="Search",
            func=search.run,
            description="useful for when you need to answer questions about current events"
        )
    ]

    # 3. Construct the Agent
    # This function replaces the deprecated 'create_react_agent'.
    # It creates a compiled LangGraph graph under the hood.
    agent_executor = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a helpful assistant. Answer the user's questions using the provided tools."
    )

    # 4. Run (Invoke the graph)
    # The input format is state-based (list of messages)
    query = "What is 2 + 2?"

    response_state = agent_executor.invoke({
        "messages": [("user", query)]
    })

    # 5. Extract and print the final response
    # The last message in the state is the AI's final answer
    final_message = response_state["messages"][-1]
    print(f"Final Answer: {final_message.content}")

except Exception as e:
    print(f"Error on running agent object: {e}")

OpenAI Chat client object created successfully.
Final Answer: The sum of 2 + 2 is 4.


In [4]:
# modern implementation using LangGraph with ChatGoogleGenerativeAI

import os
from dotenv import load_dotenv

# LangChain imports
from langchain_google_genai import ChatGoogleGenerativeAI   # NEW
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool
from langchain.agents import create_agent

load_dotenv()

try:
    print("Testing Google Generative AI client...")

    # 1. Choose the LLM --------------------------------------------------------
    # Replace ChatOpenAI with ChatGoogleGenerativeAI (Gemini 2.0 Flash)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",   # or whatever name Google stabilises on
        temperature=0,
        max_output_tokens=4096,
        google_api_key=os.getenv("GOOGLE_API_KEY")  # <— set this in .env
    )
    print("Gemini 2.0 Flash client created successfully.")

    # 2. Define Tools -----------------------------------------------------------
    if "SERPAPI_API_KEY" not in os.environ:
        raise ValueError("SERPAPI_API_KEY is missing from environment variables")

    search = SerpAPIWrapper()
    tools = [
        Tool(
            name="Search",
            func=search.run,
            description="useful for when you need to answer questions about current events"
        )
    ]

    # 3. Construct the Agent ----------------------------------------------------
    agent_executor = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a helpful assistant. Answer the user's questions using the provided tools."
    )

    # 4. Run the Agent ----------------------------------------------------------
    query = "What is 2 + 2?"
    response_state = agent_executor.invoke({"messages": [("user", query)]})

    # 5. Extract and print the final answer -------------------------------------
    final_message = response_state["messages"][-1]
    print(f"Final Answer: {final_message.content}")

except Exception as e:
    print(f"Error running agent: {e}")

Testing Google Generative AI client...
Gemini 2.0 Flash client created successfully.
Final Answer: 2 + 2 = 4


In [5]:
# Agent Examples with actual modern classes

# =============================================================================
# OLLAMA EXAMPLE WITH LANGGRAPH
# =============================================================================

"""
LangGraph Agent Examples with Multiple LLM Providers
This script demonstrates how to create agents using LangGraph with OpenAI, Claude, and Ollama
LangGraph is the modern replacement for deprecated LangChain agents
"""
import os
from dotenv import load_dotenv
from typing import Annotated, Sequence, TypedDict
import operator
from langchain_core.messages import HumanMessage, AIMessage

# LangChain imports for tools and messages
from langchain_core.messages import BaseMessage

from langchain_community.utilities.serpapi import SerpAPIWrapper

# Load environment variables
load_dotenv(override=True)


# Define the agent state
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

def setup_search_tool():
    """Initialize SerpAPI search tool"""
    try:
        serpapi_key = os.environ["SERPAPI_API_KEY"]
        search = SerpAPIWrapper(serpapi_api_key=serpapi_key)

        return Tool.from_function(
            func=search.run,
            name="search",
            description="Search the web for current information about any topic"
        )
    except KeyError:
        print("Warning: SERPAPI_API_KEY not found. Search functionality will be limited.")
        return None

# Setup search tool
search_tool = setup_search_tool()
tools = [search_tool] if search_tool else []

print("🟢 Ollama with LangGraph Example")
print("-" * 40)

try:
    from langchain_ollama import OllamaLLM

    # Initialize Ollama model
    print("Testing Ollama connection...")
    ollama_model = OllamaLLM(
        model="llama3:8b",
        base_url="http://localhost:11434",
        temperature=0
    )

    # Test basic functionality
    print("Testing Ollama llama3:8b model...")
    test_response = ollama_model.invoke("Translate 'I love programming' to Russian and explain why learning to code is valuable.")
    print(f"Ollama Response: {test_response}\n")

    # Create Ollama LangGraph agent
    if tools:
        print("🔍 Creating Ollama LangGraph Agent with Search:")

        # Note: For LLMs (not ChatModels), we need to wrap them for LangGraph
        from langchain_core.runnables import RunnableLambda

        def format_for_ollama(state):
            # Convert messages to a simple string for Ollama
            messages = state["messages"]
            if messages:
                # Get the last human message
                human_messages = [msg for msg in messages if isinstance(msg, HumanMessage)]
                if human_messages:
                    return human_messages[-1].content
            return "Hello"

        # Create a custom agent for Ollama since it's an LLM, not ChatModel
        def create_ollama_agent():
            def agent_node(state):
                query = format_for_ollama(state)

                # Check if we need to use search
                search_keywords = ["current", "latest", "recent", "news", "today", "2025", "now"]
                needs_search = any(keyword in query.lower() for keyword in search_keywords)

                if needs_search and search_tool:
                    # Use search
                    search_result = search_tool.invoke(query)
                    enhanced_query = f"Based on this search information: {search_result}\n\nNow answer the original question: {query}"
                    response = ollama_model.invoke(enhanced_query)
                else:
                    # Direct response
                    response = ollama_model.invoke(query)

                return {"messages": [AIMessage(content=response)]}

            return agent_node

        ollama_agent_node = create_ollama_agent()

        def run_ollama_agent(query: str):
            print(f"Query: {query}")
            state = {"messages": [HumanMessage(content=query)]}
            result = ollama_agent_node(state)
            return result["messages"][-1].content

        #result = run_ollama_agent("What's the current state of electric vehicle adoption in 2025?")
        result = run_ollama_agent("What professions may disappear in AI era ?")
        print(f"Ollama LangGraph Result: {result}\n")

    # Test different Ollama models
    available_models = ["mistral:7b"]

    for model_name in available_models:
        try:
            print(f"Testing {model_name} model with LangGraph...")
            alt_model = OllamaLLM(
                model=model_name,
                base_url="http://localhost:11434",
                temperature=0.3,
                timeout=30
            )

            if model_name == "codellama":
                prompt = "Write a Python function to calculate fibonacci numbers with explanation."
            elif model_name == "mistral:7b":
                prompt = "Explain machine learning concepts in simple terms."
            elif model_name == "llama3":
                prompt = "Compare Python and JavaScript for web development."
            else:
                prompt = "What is the future of artificial intelligence?"

            response = alt_model.invoke(prompt)
            print(f"{model_name.title()} Response: {response}\n")

        except Exception as e:
            print(f"❌ {model_name} model not available: {e}")

except Exception as e:
    print(f"❌ Ollama setup failed: {e}")
    print("Make sure you have:")
    print("1. Ollama installed: https://ollama.ai/")
    print("2. Ollama running: ollama serve")
    print("3. Models downloaded: ollama pull llama3:8b")
    print("4. Package installed: pip install langchain-ollama langgraph")


print("\n" + "="*60)
print("✅ LANGGRAPH EXAMPLES COMPLETE")
print("="*60)

# Summary
print("\n📋 SUMMARY:")
print("• LangGraph replaces deprecated LangChain agents")
print("• create_react_agent() - Easy agent creation for ChatModels")
print("• StateGraph - Custom workflows with multiple steps")
print("• Better control flow and state management")
print("• Works with OpenAI, Claude (ChatModels) and Ollama (LLMs)")
print("\n💡 TIP: Use create_react_agent for simple cases, StateGraph for complex workflows")

print("\n📦 REQUIRED PACKAGES:")
print("pip install langgraph langchain-openai langchain-anthropic langchain-ollama langchain-community")


🟢 Ollama with LangGraph Example
----------------------------------------
Testing Ollama connection...
Testing Ollama llama3:8b model...
Ollama Response: The translation of "I love programming" to Russian is:

Я люблю программирование (Ya lyublyu programmirovaniye)

Now, let me tell you why learning to code is valuable:

1. **High demand**: The demand for skilled programmers is extremely high and continues to grow. According to the Bureau of Labor Statistics, employment of software developers is projected to grow 21% from 2020 to 2030, much faster than the average for all occupations.
2. **Good compensation**: Programmers are well-compensated, with median salaries ranging from $60,000 to over $100,000 depending on the location and industry.
3. **Transferable skills**: Programming teaches you problem-solving, analytical thinking, and logical reasoning, which are valuable skills that can be applied to many areas of life, including business, science, and engineering.
4. **Creativity and in

In [2]:
# Agent Examples with actual modern classes

# LangGraph Agent Examples with Multiple LLM Providers
# This script demonstrates how to create agents using LangGraph with OpenAI, Claude, Gemini and Ollama
# LangGraph is the modern replacement for deprecated LangChain agents


import os
import httpx
from dotenv import load_dotenv
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool
from langchain.agents import create_agent # ✅ Modern unified entry point

# Import the specific Chat Models
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

load_dotenv()

# --- CONFIGURATION ---
# Choose your provider here: "openai", "anthropic", "google", or "ollama"
LLM_PROVIDER = "google"

http_client = httpx.Client(http2=True)

def get_llm(provider):
    """Factory function to initialize the selected LLM."""
    if provider == "openai":
        print("Initializing OpenAI (GPT-3.5 Turbo)...")
        if "OPENAI_API_KEY" not in os.environ:
            raise ValueError("OPENAI_API_KEY not found in environment variables.")
        return ChatOpenAI(model="gpt-3.5-turbo", temperature=0, http_client=http_client)

    elif provider == "anthropic":
        print("Initializing Anthropic (Claude 3.5 Haiku)...")
        if "ANTHROPIC_API_KEY" not in os.environ:
            raise ValueError("ANTHROPIC_API_KEY not found in environment variables.")
        return ChatAnthropic(model="claude-3-5-haiku-20241022", temperature=0)

    elif provider == "google":
        print("Initializing Google (Gemini 2.0 Flash)...")
        if "GOOGLE_API_KEY" not in os.environ:
            raise ValueError("GOOGLE_API_KEY not found in environment variables.")
        return ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

    elif provider == "ollama":
        print("Initializing Ollama (Mistral 7:b)...")
        # Ensure you have Ollama running locally (e.g., `ollama run llama3`)
        return ChatOllama(model="mistral:7b", temperature=0)

    else:
        raise ValueError(f"Unknown provider: {provider}")

try:
    # 1. Initialize the selected LLM
    llm = get_llm(LLM_PROVIDER)
    print(f"Initialized {LLM_PROVIDER} client.")

    # 2. Define Tools
    if "SERPAPI_API_KEY" not in os.environ:
        raise ValueError("SERPAPI_API_KEY is missing from environment variables")

    search = SerpAPIWrapper()
    tools = [
        Tool(
            name="Search",
            func=search.run,
            description="useful for when you need to answer questions about current events"
        )
    ]

    # 3. Construct the Agent
    # We use the modern 'create_agent' which works with any provider
    # that supports tool calling (OpenAI, Anthropic, Gemini, etc.)
    agent_executor = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a helpful assistant. Use the search tool if you need current information."
    )

    # 4. Run
    query = "Who is the current CEO of Microsoft and what is the stock price?"
    print(f"\nQuery: {query}\n")

    response_state = agent_executor.invoke({
        "messages": [("user", query)]
    })

    # 5. Extract Final Answer
    final_message = response_state["messages"][-1]
    print(f"Final Answer:\n{final_message.content}")

except Exception as e:
    print(f"Error: {e}")


Initializing Google (Gemini 2.0 Flash)...
Initialized google client.

Query: Who is the current CEO of Microsoft and what is the stock price?

Final Answer:
The current CEO of Microsoft is Satya Nadella. The stock price is $478.56 USD.


In [5]:
"""
LangChain create_agent Function Example
This demonstrates how to use the create_agent function from langchain.agents
"""

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool

load_dotenv()

# Define custom tools for the agent
def search_tool(query: str) -> str:
    """Simulates a search tool"""
    return f"Search results for: {query}"

def calculator_tool(expression: str) -> str:
    """Simulates a calculator tool"""
    try:
        result = eval(expression)
        return f"The result is: {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

def weather_tool(location: str) -> str:
    """Simulates a weather tool"""
    return f"The weather in {location} is sunny and 72°F"

# Create tool objects
tools = [
    Tool(
        name="Search",
        func=search_tool,
        description="Useful for searching information. Input should be a search query."
    ),
    Tool(
        name="Calculator",
        func=calculator_tool,
        description="Useful for math calculations. Input should be a mathematical expression."
    ),
    Tool(
        name="Weather",
        func=weather_tool,
        description="Useful for getting weather information. Input should be a location."
    )
]

# Initialize the LLM
if "OPENAI_API_KEY" not in os.environ:
    raise ValueError("OPENAI_API_KEY not found in environment variables.")

llm = ChatOpenAI(temperature=0, model="gpt-4")

# Create the agent using create_agent
agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the available tools to answer questions."
)

def execute_query(query: str):
    """Runs the agent and prints the final answer"""
    print(f"\nQuery: {query}\n")
    response_state = agent_executor.invoke({"messages": [("user", query)]})
    final_message = response_state["messages"][-1]
    print(f"Final Answer:\n{final_message.content}\n")

# Example usage

# Example 1: Simple calculation
print("=== Example 1: Calculation ===")
query1 = "What is 25 * 4 + 10?"
execute_query(query1)

# Example 2: Weather query
print("=== Example 2: Weather ===")
query2 = "What's the weather like in New York?"
execute_query(query2)

# Example 3: Search query
print("=== Example 3: Search ===")
query3 = "Search for information about Python programming"
execute_query(query3)

# Example 4: Multi-step task
print("=== Example 4: Multi-step ===")
query4 = "Calculate 15 * 3, then tell me the weather in that numbered street"
execute_query(query4)

# Access intermediate steps if needed
#if 'intermediate_steps' in result4:
#    print("Intermediate steps:")
#    for step in result4['intermediate_steps']:
#        print(f"  Action: {step[0]}")
#        print(f"  Result: {step[1]}")

=== Example 1: Calculation ===

Query: What is 25 * 4 + 10?

Final Answer:
The result of the calculation 25 * 4 + 10 is 110.

=== Example 2: Weather ===

Query: What's the weather like in New York?

Final Answer:
The weather in New York is sunny and 72°F.

=== Example 3: Search ===

Query: Search for information about Python programming

Final Answer:
Python is a high-level, interpreted programming language created by Guido van Rossum and first released in 1991. It's designed to be easy to read and write with its use of significant whitespace. Its language constructs and object-oriented approach aim to help programmers write clear, logical code for small and large-scale projects.

Python is dynamically typed and garbage-collected. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is often described as a "batteries included" language due to its comprehensive standard library.

Python is a popular language for web develo